# Activity 5.01: Clustering Sales Data Using K-Means
**Author:** Rim Zaher  
**GitHub Repository:** https://github.com/rimzaher-code/Assignment-7.2-Clustering-Sales-Data-Using-K-Means---Rim-Zaher
---

## 1. Project Overview & Objective
In this activity, we work with the `Sales Transaction Dataset Weekly` dataset from UCI Machine Learning Repository / Packt Publishing. The dataset contains weekly sales data for over 800 products tracked across 52 weeks (1 full year).

### Objective:
The objective is to group products with similar sales trends using the **K-Means Clustering** algorithm. Because raw sales volumes vary greatly across products, clustering is performed on normalized sales values (`Normalized 0` to `Normalized 51`) to isolate normalized sales patterns rather than absolute volume.

In [5]:
# Step 1 & 2: Load required data science libraries and ingest the dataset
import pandas as pd
import numpy as np

# Define the dataset file URL pointing directly to the raw CSV on GitHub
file_url = 'https://raw.githubusercontent.com/PacktWorkshops/The-Applied-Artificial-Intelligence-Workshop/master/Datasets/Sales_Transactions_Dataset_Weekly.csv'

# Load the dataset into a pandas DataFrame
df = pd.read_csv(file_url)

# Display DataFrame dimensions (rows = products, columns = raw sales + metadata + normalized sales)
print("Initial DataFrame shape:", df.shape)

# Inspect the top 5 rows of the dataset to verify structure
df.head()

Initial DataFrame shape: (811, 107)


,Product_Code,W0,W1,W2,W3,W4,W5,W6,W7,W8,...,Normalized 42,Normalized 43,Normalized 44,Normalized 45,Normalized 46,Normalized 47,Normalized 48,Normalized 49,Normalized 50,Normalized 51
0,P1,11,12,10,8,13,12,14,21,6,...,0.06,0.22,0.28,0.39,0.50,0.00,0.22,0.17,0.11,0.39
1,P2,7,6,3,2,7,1,6,3,3,...,0.20,0.40,0.50,0.10,0.10,0.40,0.50,0.10,0.60,0.00
2,P3,7,11,8,9,10,8,7,13,12,...,0.27,1.00,0.18,0.18,0.36,0.45,1.00,0.45,0.45,0.36
3,P4,12,8,13,5,9,6,9,13,13,...,0.41,0.47,0.06,0.12,0.24,0.35,0.71,0.35,0.29,0.35
4,P5,8,5,13,11,6,7,9,14,9,...,0.27,0.53,0.27,0.60,0.20,0.20,0.13,0.53,0.33,0.40


## 2. Feature Selection & Data Preprocessing

### Step 3 Explanation:
The dataset contains 107 columns:
* **Columns 0 to 52:** `Product_Code` and raw weekly sales counts (`W0` to `W51`).
* **Columns 53 & 54:** `MIN` and `MAX` sales values for each product.
* **Columns 55 to 106:** Normalized weekly sales values (`Normalized 0` to `Normalized 51`).

To group products based purely on normalized sales trends, we drop the first 55 columns (`Product_Code`, `W0`–`W51`, `MIN`, `MAX`) using `drop()` with `inplace=True`.

In [6]:
# Step 3: Extract the column names of the first 55 columns to drop
cols_to_drop = list(df.columns[:55])

# Drop the first 55 columns in-place to leave only the 52 normalized weekly sales features
df.drop(columns=cols_to_drop, inplace=True)

# Confirm the new shape (should be 811 rows by 52 normalized columns)
print("DataFrame shape after dropping first 55 columns:", df.shape)

# Preview the remaining normalized dataset
df.head()

DataFrame shape after dropping first 55 columns: (811, 52)


,Normalized 0,Normalized 1,Normalized 2,Normalized 3,Normalized 4,Normalized 5,Normalized 6,Normalized 7,Normalized 8,Normalized 9,...,Normalized 42,Normalized 43,Normalized 44,Normalized 45,Normalized 46,Normalized 47,Normalized 48,Normalized 49,Normalized 50,Normalized 51
0,0.44,0.50,0.39,0.28,0.56,0.50,0.61,1.00,0.17,0.61,...,0.06,0.22,0.28,0.39,0.50,0.00,0.22,0.17,0.11,0.39
1,0.70,0.60,0.30,0.20,0.70,0.10,0.60,0.30,0.30,0.30,...,0.20,0.40,0.50,0.10,0.10,0.40,0.50,0.10,0.60,0.00
2,0.36,0.73,0.45,0.55,0.64,0.45,0.36,0.91,0.82,0.27,...,0.27,1.00,0.18,0.18,0.36,0.45,1.00,0.45,0.45,0.36
3,0.59,0.35,0.65,0.18,0.41,0.24,0.41,0.65,0.65,0.53,...,0.41,0.47,0.06,0.12,0.24,0.35,0.71,0.35,0.29,0.35
4,0.33,0.13,0.67,0.53,0.20,0.27,0.40,0.73,0.40,0.40,...,0.27,0.53,0.27,0.60,0.20,0.20,0.13,0.53,0.33,0.40


## 3. K-Means Model Training & Cluster Assignment

### Steps 4 & 5 Explanation:
1. We initialize a `KMeans` model with `n_clusters=8` and `random_state=8` to ensure deterministic, reproducible results.
2. We fit the model on our normalized feature set (`df`).
3. We extract the resulting cluster assignment array (`labels_`) corresponding to each product row.

In [7]:
# Import the KMeans clustering module from scikit-learn
from sklearn.cluster import KMeans

# Step 4: Create a k-means clustering model with 8 clusters and random_state = 8
kmeans = KMeans(n_clusters=8, random_state=8)

# Fit the k-means model on the normalized sales DataFrame
kmeans.fit(df)

# Step 5: Retrieve the cluster labels assigned to each product
labels = kmeans.labels_

# Print the distribution of products across the 8 generated clusters
print("Product counts per cluster:")
print(pd.Series(labels).value_counts().sort_index())

Product counts per cluster:
0      8
1     90
2     96
3    168
4     85
5    128
6    129
7    107
Name: count, dtype: int64


## 4. Aggregating Raw Weekly Sales by Cluster

### Steps 6 & 7 Explanation:
1. To inspect the original sales scale for each cluster, we reload the raw dataset and extract only the 52 weekly sales columns (`W0` through `W51`).
2. We add the `cluster` label column obtained from the K-Means model to this raw sales DataFrame.
3. We apply `groupby('cluster').mean()` to calculate the average weekly sales across all products in each cluster across the full 52-week year.

In [8]:
# Step 6a: Re-read the original raw dataset from the source URL
raw_df = pd.read_csv(file_url)

# Step 6b: Extract list of weekly raw sales column names W0 to W51
w_cols = [f'W{i}' for i in range(52)]

# Step 6c: Filter raw DataFrame to keep only the W columns
df_w = raw_df[w_cols].copy()

# Step 6d: Add the cluster labels as a new column named 'cluster'
df_w['cluster'] = labels

# Step 7: Perform aggregation using groupby to obtain yearly average sales per cluster
cluster_summary = df_w.groupby('cluster').mean()

# Display the final aggregated DataFrame (Matches expected output Figure 5.7)
cluster_summary

,W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,...,W42,W43,W44,W45,W46,W47,W48,W49,W50,W51
cluster,,,,,,,,,,,,,,,,,,,,,
0,0.000000,0.375000,0.125000,0.250000,0.000000,0.000000,0.625000,0.125000,0.125000,0.500000,...,0.125000,0.250000,0.000000,0.875000,0.500000,0.125000,0.000000,0.500000,0.000000,0.125000
1,7.988889,8.477778,8.488889,8.477778,9.288889,9.100000,8.411111,8.822222,9.933333,9.366667,...,9.444444,9.000000,8.633333,8.566667,9.066667,9.811111,9.811111,11.722222,12.044444,14.488889
2,10.562500,10.427083,9.802083,10.385417,10.187500,11.031250,10.760417,10.635417,10.177083,9.416667,...,8.614583,8.083333,8.312500,8.531250,8.781250,9.229167,9.239583,10.114583,11.406250,12.666667
3,0.077381,0.130952,0.101190,0.077381,0.107143,0.107143,0.083333,0.101190,0.047619,0.119048,...,0.214286,0.184524,0.226190,0.184524,0.315476,0.154762,0.327381,0.303571,0.261905,0.464286
4,3.388235,2.988235,3.282353,3.729412,3.635294,3.505882,4.129412,4.105882,3.623529,3.752941,...,3.635294,3.847059,4.200000,3.729412,4.141176,4.717647,4.011765,4.458824,5.247059,7.141176
5,26.492188,28.148438,29.054688,29.218750,29.132812,28.445312,30.187500,28.609375,30.085938,29.445312,...,24.945312,24.781250,24.984375,25.257812,25.343750,24.773438,24.710938,23.695312,22.351562,18.296875
6,13.558140,13.317829,14.263566,15.511628,14.155039,13.868217,13.961240,14.612403,14.271318,15.046512,...,11.674419,11.930233,12.147287,12.713178,12.651163,12.046512,12.147287,12.124031,11.302326,10.829457
7,0.429907,0.373832,0.504673,0.448598,0.644860,0.495327,0.542056,0.420561,0.514019,0.485981,...,0.803738,0.831776,0.981308,1.130841,1.196262,1.028037,1.299065,1.467290,1.859813,2.467290
